# V7.5 — FRESH HARD-NEGATIVE SAFETY TEST — SINGLE HUMAN REVIEWER

**Dữ liệu đã được một người gán nhãn thủ công:** 126 ABSENT và 34 PRESENT trong 160 cặp. Chỉ 126 ABSENT được đưa vào thử nghiệm đếm với ground truth = 0; 34 PRESENT bị loại khỏi phép đo negative FPR. **Không có đồng thuận reviewer thứ hai.**

Inputs Kaggle: dataset ảnh FSC-147 `hihihi5`, file `V75_FreshNeg_Single_Reviewer_Inputs.zip` từ bộ tải về này. Bật GPU + Internet. Chạy **Run All** sau khi Add Input.

Giữ nguyên cấu hình V7.5 (CountGD++ threshold, tiling 4/8/12, Qwen, gating). Không điều chỉnh ngưỡng theo kết quả test. Notebook chỉ đo **độ an toàn của nhánh multi-scale rescue** so với CountGD++ trên 126 negative mới; **không** đánh giá toàn bộ pipeline V7 + V7.5, vì V7 dual-evidence router không được chạy ở đây.

**Báo cáo khoa học:** Ghi rõ “single-annotator manually screened hard-negative set; no independent adjudication”. Một số nhãn có thể sai vì đối tượng gần nghĩa/khó phân biệt; đây là giới hạn của benchmark.

In [ ]:
# 01 — FROZEN CONFIG, NO ACCESS TO LABELS
import os,sys,time,json,re,math,io,gc,zipfile,shutil,hashlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageOps
SEED=42; np.random.seed(SEED)
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT=ROOT/'v75_freshneg_safety';OUT.mkdir(parents=True,exist_ok=True)
INPUT_ROOTS=[r for r in [Path('/kaggle/input'),Path('/mnt/data')] if r.exists()]
DATA_DIR=OUT/'inputs';DATA_DIR.mkdir(exist_ok=True)
BOX_DIR=OUT/'tiling_boxes';BOX_DIR.mkdir(exist_ok=True)
BASE_CKPT=OUT/'freshneg_baseline.csv';TILE_CKPT=OUT/'freshneg_tiles.csv';GATE_CKPT=OUT/'freshneg_qwen_gate.csv'
COUNTGD_REPO=ROOT/'CountGDPlusPlus'
COUNTGD_GITHUB='https://github.com/niki-amini-naieni/CountGDPlusPlus.git'
COUNTGD_GDRIVE_ID='1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs'
QWEN_MODEL_ID='Qwen/Qwen3-VL-4B-Instruct'
COUNTGD_THRESH=.23
TILE_OVERLAP_FRAC=.20
NMS_IOU=.50
N_SAMPLE=6
MIN_LOCAL_YES=5
CONTEXT_MULTIPLIER=6.0
PANEL_SIZE=336
TRIGGER_BASELINE_MAX=3
MIN_TILED=6
MAX_SCALE_RATIO=2.5
FINE_GRIDS=(4,8,12)
# Keep this exact protocol for REPLICATION_A and REPLICATION_B. No threshold tuning mid-run.
PROTOCOL={'threshold':COUNTGD_THRESH,'overlap':TILE_OVERLAP_FRAC,'nms_iou':NMS_IOU,
 'trigger_baseline_le':TRIGGER_BASELINE_MAX,'grids':FINE_GRIDS,'min_grid8':MIN_TILED,
 'min_local_yes':MIN_LOCAL_YES,'max_scale_ratio':MAX_SCALE_RATIO,
 'independent_full_image_qwen_yes':True,'sample_seed':'V75_PILOT_BLIND_2026_11_03',
 'model':QWEN_MODEL_ID,'evaluation':'single-annotator manually screened NEW FSC147 val negatives; FROZEN V75 rescue branch only',
 'label_provenance':'one human annotator; no independent double check; 34 PRESENT excluded'}
(OUT/'freshneg_frozen_protocol.json').write_text(json.dumps(PROTOCOL,indent=2))
print('OUTPUT',OUT)

MIN_PAIRS=100  # current 126 single-reviewed ABSENT; no second reviewer required

In [ ]:
# 02 — LOAD REVIEWED NEW NEGATIVES (NO IMAGE COUNTS / GT READ)
def find_input_file(filename):
    for root in INPUT_ROOTS:
        hits=sorted(root.rglob(filename),key=str)
        if hits:return hits[0]
    return None

def get_input_file(filename):
    dest=DATA_DIR/filename
    if dest.exists():return dest
    existing=find_input_file(filename)
    if existing:
        shutil.copyfile(existing,dest);return dest
    for root in INPUT_ROOTS:
        for zpath in sorted(root.rglob('*.zip'),key=str):
            try:
                with zipfile.ZipFile(zpath) as z:
                    possible=[name for name in z.namelist() if Path(name).name==filename]
                    if possible:
                        dest.write_bytes(z.read(possible[0]));return dest
            except (zipfile.BadZipFile,PermissionError):pass
    raise FileNotFoundError('Missing reviewed CSV: '+filename+'; Add Input a manually reviewed file')

reviewed=pd.read_csv(get_input_file('V75_FreshNeg_Reviewed_160_SINGLE_REVIEWER.csv'),keep_default_na=False)
candidates=pd.read_csv(get_input_file('V75_FreshNeg_Candidates_160.csv'),keep_default_na=False)
join_key=['candidate_id','group_id','image_name','source_class','target','candidate_type','eval_partition']
assert all(k in reviewed.columns for k in join_key)
assert len(candidates)==160 and candidates.group_id.is_unique
assert reviewed.group_id.is_unique and len(reviewed)==160
chk=reviewed[join_key].sort_values('group_id').reset_index(drop=True)
expected=candidates[join_key].sort_values('group_id').reset_index(drop=True)
assert chk.equals(expected), 'Reviewed manifest altered candidate IDs / target / image; edit ONLY review columns.'
for c in ['reviewer_1_decision','reviewer_2_decision','adjudicated_presence']:
    reviewed[c]=reviewed[c].astype(str).str.strip().str.upper()
    assert set(reviewed[c]).issubset({'','ABSENT','PRESENT','UNCERTAIN'}), 'Unexpected review label in '+c
# SINGLE-ANNOTATOR PROTOCOL: use ONLY reviewer's explicit manual decisions.
assert reviewed.reviewer_1_decision.isin({'ABSENT','PRESENT','UNCERTAIN'}).all(), 'Reviewer 1 must label every pair.'
assert reviewed.adjudicated_presence.equals(reviewed.reviewer_1_decision), \
    'Compatibility label must match reviewer 1; this is NOT consensus adjudication.'
assert reviewed.reviewer_2_decision.eq('').all(), \
    'Single-reviewer evaluation must not imply an independent second reviewer.'
verified=reviewed.loc[reviewed.reviewer_1_decision.eq('ABSENT')].copy()
assert len(verified)>=MIN_PAIRS, 'Not enough manually ABSENT pairs: '+str(len(verified))
verified['double_checked']=False  # provenance: always FALSE
print('SINGLE reviewer ABSENT:',len(verified), '| PRESENT excluded:',
      int(reviewed.reviewer_1_decision.eq('PRESENT').sum()),
      '| UNCERTAIN excluded:',int(reviewed.reviewer_1_decision.eq('UNCERTAIN').sum()))
print('NO second annotator. This is exploratory safety evaluation, NOT consensus GT.')
work=verified[['group_id','image_name','target','source_class','candidate_type','eval_partition','double_checked']].copy()
assert not any('gt' in k or k.endswith('_count') for k in work.columns)
img_root=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
image_map={}
for p in img_root.rglob('*.jpg'):
    image_map.setdefault(p.name,str(p))
missing=set(work.image_name)-set(image_map)
if missing:raise FileNotFoundError('Add FSC147 hihihi5 input; missing JPG: '+str(sorted(missing)[:10]))
work['image_path']=work.image_name.map(image_map)
# Use one and only one verified reviewed set per experiment. Make it content-addressed.
roster_bytes=work[['group_id','target']].sort_values('group_id').to_csv(index=False).encode()
ROSTER_SHA=hashlib.sha256(roster_bytes).hexdigest()
(OUT/'freshneg_review_roster_sha256.txt').write_text(ROSTER_SHA+'\n')
print('Review roster SHA256:',ROSTER_SHA)

In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision hiện tại.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v4_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers==4.57.1",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# 02B — REPAIR HUGGING FACE TRANSFORMERS INSTALLATION BEFORE COUNTGD++
# Fix for: ValueError: Offending backend: tensorflow_text
# Do NOT install tensorflow-text. Keep Torch/Torchvision unchanged.
import os, sys, site, sysconfig, subprocess, importlib, shutil
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

PINNED_TRANSFORMERS = "4.57.1"
probe_code = (
    'import os; os.environ["USE_TF"]="0"; os.environ["USE_FLAX"]="0"; '
    'import transformers; '
    'from transformers import BertModel, BertTokenizer, Qwen3VLForConditionalGeneration; '
    'print("OK transformers", transformers.__version__)'
)

def test_import():
    return subprocess.run(
        [sys.executable, "-c", probe_code],
        text=True, capture_output=True,
        env={**os.environ, "USE_TF": "0", "USE_FLAX": "0"},
    )

check = test_import()
if check.returncode != 0:
    print("Transformers import failed; cleaning old package files ...")
    print("\n".join((check.stderr or "").splitlines()[-7:]))
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "transformers"],
        check=True,
    )
    # Pip uninstall may leave obsolete files (e.g. tokenization_bert_tf.py).
    # ONLY remove the transformers package folder in Python site-packages.
    site_roots = set(site.getsitepackages())
    site_roots.add(sysconfig.get_paths()["purelib"])
    site_roots.add(site.getusersitepackages())
    for root in sorted(site_roots):
        folder = Path(root) / "transformers"
        if folder.is_dir() and folder.parent.name in ("site-packages", "dist-packages"):
            print("Removing stale files:", folder)
            shutil.rmtree(folder)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
         "--no-deps", f"transformers=={PINNED_TRANSFORMERS}"],
        check=True,
    )
    importlib.invalidate_caches()
    # Clear partial module objects from a prior failed lazy import.
    for mod in list(sys.modules):
        if mod == "transformers" or mod.startswith("transformers."):
            del sys.modules[mod]
    check = test_import()

if check.returncode != 0:
    print("\n".join((check.stderr or "").splitlines()[-28:]))
    raise RuntimeError("Transformers vẫn không import được; gửi log của cell này.")
print("PASS:", check.stdout.strip())
print("Torch/Torchvision were not reinstalled.")


In [ ]:
# ============================================================
# CELL 3 — VERIFY TORCH/CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import gc
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V3 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS", MSDA.__file__)


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


In [ ]:
# 05B — RESET TRANSFORMERS smart_apply WHEN NOTEBOOK SWITCHED VERSIONS
# CountGD++ uses BertModel.from_pretrained; transformers 4.57.1 expects
# smart_apply(self, fn). A newer Kaggle version may have previously defined
# smart_apply(self, fn, is_custom_code) on torch.nn.Module.
# This temporary method survives pip reinstall in an active Python kernel.
import torch, inspect, transformers
print('Current transformers:', transformers.__version__)
if transformers.__version__ != '4.57.1':
    raise RuntimeError(
        'Notebook V7.4 expects transformers==4.57.1. '
        'Please rerun the installation/repair cell before loading CountGD++.'
    )
if 'smart_apply' in vars(torch.nn.Module):
    print('Removing previous smart_apply signature:',
          inspect.signature(torch.nn.Module.smart_apply))
    delattr(torch.nn.Module, 'smart_apply')
else:
    print('No stale smart_apply found.')
print('PASS: BertModel will register matching smart_apply on next load.')


In [ ]:
# 06 — LOAD COUNTGD++ (giữ nguyên API cũ)
import torch, importlib
# Reset stale Transformers smart_apply, e.g. new-version (self, fn, is_custom_code)
# persists in torch.nn.Module after pip installs transformers 4.57.1.
if 'smart_apply' in vars(torch.nn.Module):
    delattr(torch.nn.Module, 'smart_apply')
import gc
p=str(COUNTGD_REPO)
if p not in sys.path:sys.path.insert(0,p)
os.chdir(str(COUNTGD_REPO))
for name in list(sys.modules):
    if name=='models' or name.startswith('models.'):
        del sys.modules[name]
importlib.invalidate_caches();gc.collect();torch.cuda.empty_cache()
process_file=COUNTGD_REPO/'process_folder.py'
source=process_file.read_text(encoding='utf-8')
marker='logging.basicConfig(level=logging.INFO'
if marker not in source:raise RuntimeError('API CountGD++ thay đổi')
countgd_namespace={'__name__':'v72_countgd_runtime','__file__':str(process_file)}
exec(compile(source.split(marker,1)[0],str(process_file),'exec'),countgd_namespace)
for k in ['build_arg_parser','build_model_and_transforms','count']:
    if k not in countgd_namespace:raise RuntimeError(f'Missing CountGD++ API: {k}')
args=countgd_namespace['build_arg_parser']().parse_args([])
args.device='cuda:0';args.pretrain_model_path=str(COUNTGD_CKPT)
_load=torch.load
def _load_trusted(*a,**kw):
    kw.setdefault('weights_only',False)
    return _load(*a,**kw)
torch.load=_load_trusted
try:
    countgd_model,countgd_transform=countgd_namespace['build_model_and_transforms'](args)
finally:
    torch.load=_load
countgd_model=countgd_model.to('cuda:0').eval()
countgd_namespace.update(model=countgd_model,transform=countgd_transform,device=torch.device('cuda:0'))
print('CountGD++ loaded')


In [ ]:
# 07 — HELPERS: TILE, TỌA ĐỘ TOÀN ẢNH, NMS, CHECKPOINT

def atomic_csv(df,path):
    temp=Path(str(path)+'.tmp')
    df.to_csv(temp,index=False)
    temp.replace(path)

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    inter=max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1)
    area_b=max(0,bx2-bx1)*max(0,by2-by1)
    return inter/max(1e-9,area_a+area_b-inter)

def nms_rows(rows,thr=NMS_IOU):
    chosen=[]
    for row in sorted(rows,key=lambda r:-float(r['score'])):
        if not any(iou(row['box'],prev['box'])>thr for prev in chosen):
            chosen.append(row)
    return chosen

def count_tiled(img,target,grid):
    W,H=img.size
    detections=[]
    for gy in range(grid):
        for gx in range(grid):
            core_x0=int(round(gx*W/grid));core_x1=int(round((gx+1)*W/grid))
            core_y0=int(round(gy*H/grid));core_y1=int(round((gy+1)*H/grid))
            dx=(core_x1-core_x0)*TILE_OVERLAP_FRAC
            dy=(core_y1-core_y0)*TILE_OVERLAP_FRAC
            x0=max(0,int(math.floor(core_x0-dx)))
            x1=min(W,int(math.ceil(core_x1+dx)))
            y0=max(0,int(math.floor(core_y0-dy)))
            y1=min(H,int(math.ceil(core_y1+dy)))
            crop=img.crop((x0,y0,x1,y1))
            if min(crop.size)<8:continue
            boxes,scores,_=countgd_namespace['count'](crop,str(target).replace('.','').strip(),{'image':crop,'points':[]},[],[],float(COUNTGD_THRESH),False)
            boxes=boxes.tolist() if hasattr(boxes,'tolist') else list(boxes)
            scores=scores.tolist() if hasattr(scores,'tolist') else list(scores)
            for box,score in zip(boxes,scores):
                score=float(score)
                if score<=COUNTGD_THRESH:continue
                xx0,yy0,xx1,yy1=[float(v) for v in box[:4]]
                # CountGD++ gốc trả bbox xyxy theo pixel của PIL image.
                xx0=min(max(xx0+x0,0),W);xx1=min(max(xx1+x0,0),W)
                yy0=min(max(yy0+y0,0),H);yy1=min(max(yy1+y0,0),H)
                bx0,bx1=sorted([xx0,xx1]);by0,by1=sorted([yy0,yy1])
                cx=(bx0+bx1)/2;cy=(by0+by1)/2
                # Ownership by center, tránh đếm hai lần ở các ô overlap.
                inside_x = core_x0<=cx<(core_x1 if gx<grid-1 else core_x1+1)
                inside_y = core_y0<=cy<(core_y1 if gy<grid-1 else core_y1+1)
                if inside_x and inside_y and bx1>bx0 and by1>by0:
                    detections.append({'box':[bx0,by0,bx1,by1],'score':score,'grid':grid,'cell':[gx,gy]})
    return nms_rows(detections)

# small self-test of NMS and IoU
assert iou([0,0,10,10],[0,0,10,10])>0.999
assert len(nms_rows([{'box':[0,0,10,10],'score':.9},{'box':[1,1,10,10],'score':.8}]))==1
print('Helpers ready; NMS self-test PASS')

In [ ]:
# 10 — CHECKPOINT ENGINE; RESTORE PREVIOUS KAGGLE OUTPUT IF AVAILABLE

def snapshot():
    outzip=ROOT/'V75_FreshNeg_Partial_Checkpoint.zip'
    tmp=outzip.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in (BASE_CKPT,TILE_CKPT,GATE_CKPT):
            if p.exists():z.write(p,p.name)
        for p in BOX_DIR.glob('*.json'):z.write(p,'tiling_boxes/'+p.name)
        z.write(OUT/'freshneg_frozen_protocol.json','freshneg_frozen_protocol.json')
        z.write(OUT/'freshneg_review_roster_sha256.txt','freshneg_review_roster_sha256.txt')
    tmp.replace(outzip)
    return outzip

def restore_checkpoint():
    zp=find_input_file('V75_FreshNeg_Partial_Checkpoint.zip')
    if not zp:return
    print('Khôi phục checkpoint:',zp)
    with zipfile.ZipFile(zp) as z:
        if 'freshneg_frozen_protocol.json' in z.namelist():
            old=json.loads(z.read('freshneg_frozen_protocol.json'))
            assert old==PROTOCOL, 'PROTOCOL đã thay đổi, không thể trộn checkpoint'
        if 'freshneg_review_roster_sha256.txt' in z.namelist():
            assert z.read('freshneg_review_roster_sha256.txt').decode().strip()==ROSTER_SHA, 'Roster changed: restart into a new experiment, do not mix predictions'
        for src,dst in [('freshneg_baseline.csv',BASE_CKPT),('freshneg_tiles.csv',TILE_CKPT),('freshneg_qwen_gate.csv',GATE_CKPT)]:
            if not dst.exists() and src in z.namelist():dst.write_bytes(z.read(src))
        for name in z.namelist():
            if name.startswith('tiling_boxes/') and name.endswith('.json'):
                dst=BOX_DIR/Path(name).name
                if not dst.exists():dst.write_bytes(z.read(name))

restore_checkpoint()

def read_checkpoint(path):
    if not path.exists():return []
    return pd.read_csv(path).to_dict('records')

def upsert_record(records,rec,path):
    records=[x for x in records if str(x['group_id'])!=rec['group_id']]+[rec]
    atomic_csv(pd.DataFrame(records),path)
    snapshot()
    return records
print('Checkpoint engine ready')

In [ ]:
# 11 — COUNTGD++ FULL-FRAME BASELINE FOR EVERY VAL PILOT IMAGE
rows=read_checkpoint(BASE_CKPT)
done={str(r['group_id']) for r in rows if r.get('status')=='ok'}
for i,r in enumerate(work.itertuples(index=False),1):
    if r.group_id in done:continue
    t0=time.time()
    try:
        img=Image.open(r.image_path).convert('RGB')
        boxes,scores,_=countgd_namespace['count'](img,str(r.target).replace('.','').strip(),
                {'image':img,'points':[]},[],[],COUNTGD_THRESH,False)
        # Match the baseline convention used by tiling code; CountGD count returns model boxes.
        n=len(boxes)
        rec={'group_id':r.group_id,'baseline_count':int(n),'status':'ok',
             'elapsed_sec':round(time.time()-t0,2),'error':''}
    except Exception as e:
        rec={'group_id':r.group_id,'baseline_count':-1,'status':'error',
             'elapsed_sec':round(time.time()-t0,2),'error':repr(e)}
    rows=upsert_record(rows,rec,BASE_CKPT)
    print(f'[BASE] {i}/{len(work)} {r.group_id}: {rec["baseline_count"]} ({rec["status"]})')
    if rec['status']!='ok':raise RuntimeError('BASELINE failed: '+rec['error'])
assert len(pd.read_csv(BASE_CKPT).query('status=="ok"'))==len(work)
base_counts=pd.read_csv(BASE_CKPT).set_index('group_id').baseline_count.to_dict()
triggers=work.loc[work.group_id.map(base_counts).le(TRIGGER_BASELINE_MAX)].copy()
print('Trigger eligible (independent of GT):',len(triggers))


In [ ]:
# 12 — DYNAMIC 4x4 → 8x8 → OPTIONAL 12x12 TILING, WITH CHECKPOINT
rows=read_checkpoint(TILE_CKPT)
done={str(r['group_id']) for r in rows if r.get('status')=='ok'}
for i,r in enumerate(triggers.itertuples(index=False),1):
    if r.group_id in done:continue
    t0=time.time()
    data={'grid4':[],'grid8':[],'grid12':[]}
    try:
        img=Image.open(r.image_path).convert('RGB')
        data['grid4']=count_tiled(img,r.target,4)
        data['grid8']=count_tiled(img,r.target,8)
        # 12x12 runs only with enough evidence from 8x8; no ground truth used
        if len(data['grid8'])>=MIN_TILED:
            data['grid12']=count_tiled(img,r.target,12)
        (BOX_DIR/(r.group_id+'.json')).write_text(json.dumps(data),encoding='utf-8')
        rec={'group_id':r.group_id,'grid4':len(data['grid4']),
             'grid8':len(data['grid8']),'grid12':len(data['grid12']),
             'ran_grid12':len(data['grid8'])>=MIN_TILED,
             'status':'ok','elapsed_sec':round(time.time()-t0,2),'error':''}
    except Exception as e:
        rec={'group_id':r.group_id,'grid4':len(data['grid4']),
             'grid8':len(data['grid8']),'grid12':len(data['grid12']),
             'ran_grid12':False,'status':'error',
             'elapsed_sec':round(time.time()-t0,2),'error':repr(e)}
    rows=upsert_record(rows,rec,TILE_CKPT)
    print(f'[TILES] {i}/{len(triggers)} {r.group_id}:',rec['grid4'],rec['grid8'],rec['grid12'])
    if rec['status']!='ok':raise RuntimeError('TILING failed: '+rec['error'])
assert len(pd.read_csv(TILE_CKPT).query('status=="ok"'))==len(triggers) if len(triggers) else True
print('Multi-scale stage complete')


In [ ]:
# 09 — free CountGD++ before loading Qwen3-VL
import torch,gc
for name in ('countgd_model','countgd_transform'):
    if name in globals():del globals()[name]
if 'countgd_namespace' in globals():
    countgd_namespace.pop('model',None)
    countgd_namespace.pop('transform',None)
gc.collect();torch.cuda.empty_cache()
print('CUDA cache cleared')

In [ ]:
# CELL 3 — QWEN3-VL: LOAD PROCESSOR FROM LOCAL CONFIG FILES
# Avoid 404 on HF Hub optional additional_chat_templates directory.
# Keep Transformers 4.57.1: do not upgrade it in this cell.
import torch, gc, sys, subprocess
from huggingface_hub import snapshot_download
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle Session options."
try:
    from qwen_vl_utils import process_vision_info
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils'],
        check=True,
    )
    from qwen_vl_utils import process_vision_info

# Only small processor/tokenizer files, NOT 9GB model weights.
processor_dir = snapshot_download(
    repo_id=QWEN_MODEL_ID,
    allow_patterns=[
        'config.json',
        'processor_config.json',
        'preprocessor_config.json',
        'video_preprocessor_config.json',
        'tokenizer.json',
        'tokenizer_config.json',
        'vocab.json',
        'merges.txt',
        'chat_template.json',
        'chat_template.jinja',
        'special_tokens_map.json',
        'added_tokens.json',
    ],
)

# A local path prevents unnecessary HF Hub queries for a missing
# optional folder named additional_chat_templates.
processor = AutoProcessor.from_pretrained(
    processor_dir,
    local_files_only=True,
)
print('Qwen processor loaded from:', processor_dir)

# Download/load model weights normally; model ID is valid.
model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
).eval()
print('Loaded Qwen:', QWEN_MODEL_ID, 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
# CELL 4 — CONTEXT PANELS & OUTPUT PARSER (unit tests)
from PIL import Image, ImageDraw, ImageOps

LINE_RX=re.compile(r'^(?:ID\s*)?(\d+)\s*[|:]\s*(YES|NO|UNCERTAIN)\s*$',re.I)
FULL_RX=re.compile(r'\b(?:YES|NO|UNCERTAIN)\b',re.I)

def parse_full(text):
    tags=FULL_RX.findall(str(text).upper())
    return tags[0] if len(tags)==1 else 'UNCERTAIN'

def parse_panel(text,n):
    out={}
    for line in str(text).strip().splitlines():
        match=LINE_RX.match(line.strip())
        if match:
            idx=int(match.group(1))
            if 0<=idx<n and idx not in out:out[idx]=match.group(2).upper()
    return out

def choose_diverse(boxes,n=N_SAMPLE):
    boxes=sorted(boxes,key=lambda x:-float(x.get('score',0)))
    chosen=[];used_cells=set()
    for x in boxes:
        cell=tuple(x.get('cell',[0,0]))
        if cell not in used_cells:
            chosen.append(x);used_cells.add(cell)
        if len(chosen)>=n:return chosen
    used_ids={id(x) for x in chosen}
    for x in boxes:
        if id(x) not in used_ids:
            chosen.append(x)
            if len(chosen)>=n:break
    return chosen

def local_context_box(im_size, bbox, mult=CONTEXT_MULTIPLIER):
    W,H=im_size
    x1,y1,x2,y2=map(float,bbox)
    cx=(x1+x2)/2;cy=(y1+y2)/2
    bw=max(1,x2-x1);bh=max(1,y2-y1)
    crop_w=max(mult*bw,40);crop_h=max(mult*bh,40)
    # Ít nhất một bề rộng nhỏ vừa đủ nhìn rõ đối tượng nhưng không phóng quá 50% ảnh.
    crop_w=min(W,max(crop_w,2*bw));crop_h=min(H,max(crop_h,2*bh))
    left=max(0,min(W-crop_w,cx-crop_w/2));top=max(0,min(H-crop_h,cy-crop_h/2))
    return tuple(map(int,[round(left),round(top),round(left+crop_w),round(top+crop_h)]))

def sheet_context(image, chosen):
    n=len(chosen); cols=3;rows=(n+cols-1)//cols; head=32
    sheet=Image.new('RGB',(cols*PANEL_SIZE,rows*(PANEL_SIZE+head)),'white')
    draw=ImageDraw.Draw(sheet)
    for i,one in enumerate(chosen):
        box=one['box'];rect=local_context_box(image.size,box)
        crop=image.crop(rect).convert('RGB')
        panel=ImageOps.contain(crop,(PANEL_SIZE-12,PANEL_SIZE-12))
        x0=(i%cols)*PANEL_SIZE;y0=(i//cols)*(PANEL_SIZE+head)
        ox=x0+(PANEL_SIZE-panel.width)//2;oy=y0+(PANEL_SIZE-panel.height)//2
        sheet.paste(panel,(ox,oy))
        # Convert global box to panel space (contain preserves aspect ratio)
        scale=panel.width/max(1,crop.width)
        bx=[ox+(float(box[j])-rect[0])*scale if j%2==0 else oy+(float(box[j])-rect[1])*scale for j in range(4)]
        draw.rectangle(bx,outline='red',width=4)
        draw.rectangle((x0,y0+PANEL_SIZE,x0+PANEL_SIZE-1,y0+PANEL_SIZE+head-1),fill='white',outline='black',width=1)
        draw.text((x0+8,y0+PANEL_SIZE+5),f'ID {i} (red rectangle)',fill='black')
    return sheet

assert parse_full('YES')=='YES'
assert parse_full('NO')=='NO'
assert parse_full('YES, NO')=='UNCERTAIN'
assert parse_panel('0|YES\n1|NO\n2|UNCERTAIN',3)=={0:'YES',1:'NO',2:'UNCERTAIN'}
assert local_context_box((500,400),[10,10,20,20])[2]>10
print('Context + parser self-tests PASS')

In [ ]:
# 15 — INDEPENDENT FULL-IMAGE + LOCAL PANEL VERIFICATION (NO GT)
@torch.inference_mode()
def qwen_infer(messages,max_tokens=128):
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    images,videos,kwargs=process_vision_info(messages,image_patch_size=16,
                                      return_video_kwargs=True,return_video_metadata=True)
    metas=None
    if videos is not None:
        videos,metas=zip(*videos);videos=list(videos);metas=list(metas)
    inputs=processor(text=[prompt],images=images,videos=videos,video_metadata=metas,
                     padding=True,return_tensors='pt',do_resize=False,**kwargs)
    inputs=inputs.to(next(model.parameters()).device)
    output=model.generate(**inputs,max_new_tokens=max_tokens,do_sample=False)
    return processor.batch_decode([a[len(b):] for b,a in zip(inputs.input_ids,output)],
                  skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]

@torch.inference_mode()
def full_presence(image,target):
    prompt=(f'Target category: "{target}". Does at least one genuine, individual instance of this '
            'exact category appear in the photograph? Ignore background textures, parts of an object, '
            'and similar-looking categories. If not certain, answer UNCERTAIN. '
            'Reply with ONLY YES, NO, or UNCERTAIN.')
    return qwen_infer([{'role':'user','content':[{'type':'image','image':image},
                                      {'type':'text','text':prompt}]}],max_tokens=24)

@torch.inference_mode()
def local_confirmation(image,target,chosen):
    sheet=sheet_context(image,chosen)
    prompt=(f'Target category: "{target}". Image 1 is the original scene. '
      'Image 2 shows SIX numbered crops with RED rectangles. For each crop classify ONLY '
      'the INSTANCE INSIDE its red rectangle, not the surrounding context. '
      'Do not count patterns, pieces, studs, shadows or related categories as the target. '
      'If ambiguous respond UNCERTAIN. Output exactly six lines: 0|YES, 1|NO, '
      '2|UNCERTAIN, etc., one label per ID, with no explanation.')
    raw=qwen_infer([{'role':'user','content':[{'type':'image','image':image},
                                           {'type':'image','image':sheet},
                                           {'type':'text','text':prompt}]}],max_tokens=128)
    return raw,sheet

assert parse_full('YES')=='YES'
assert parse_full('NO')=='NO'
assert parse_panel('0|YES\n1|NO',2)=={0:'YES',1:'NO'}
print('Full-image and panel parsing: PASS')


In [ ]:
# 16 — FIXED ROUTER, CHECKPOINT AFTER EACH QWEN IMAGE
rows=read_checkpoint(GATE_CKPT)
done={str(r['group_id']) for r in rows if r.get('status')=='ok'}
tiles_by_gid=pd.read_csv(TILE_CKPT).set_index('group_id').to_dict('index') if TILE_CKPT.exists() else {}
for i,r in enumerate(triggers.itertuples(index=False),1):
    if r.group_id in done:continue
    t0=time.time(); n0=int(base_counts[r.group_id]);tile_row=tiles_by_gid[r.group_id]
    n4=int(tile_row['grid4']);n8=int(tile_row['grid8']);n12=int(tile_row['grid12'])
    approved=False; full_label='NOT_RUN'; local_yes=0;panel_complete=False; raw_full='';raw_panel=''
    new_count=n0
    try:
        # No Qwen needed without 2 viable tile scales, and no class evidence from GT.
        if n8>=MIN_TILED and n12>=MIN_TILED:
            img=Image.open(r.image_path).convert('RGB')
            raw_full=full_presence(img,r.target)
            full_label=parse_full(raw_full)
            if full_label=='YES':
                stored=json.loads((BOX_DIR/(r.group_id+'.json')).read_text())
                # Select from 12x12 results; if fewer than six boxes, no change.
                chosen=choose_diverse(stored['grid12'],N_SAMPLE)
                if len(chosen)==N_SAMPLE:
                    raw_panel,sheet=local_confirmation(img,r.target,chosen)
                    sheet.save(OUT/(r.group_id+'_v75rep_context.jpg'),quality=87)
                    parsed=parse_panel(raw_panel,N_SAMPLE)
                    panel_complete=len(parsed)==N_SAMPLE
                    local_yes=sum(x=='YES' for x in parsed.values())
            # Two scales agree within a fixed maximum multiplicative difference.
            ratio=max(n8,n12)/max(1,min(n8,n12))
            approved=(full_label=='YES' and panel_complete and local_yes>=MIN_LOCAL_YES
                    and ratio<=MAX_SCALE_RATIO)
            if approved:
                proposal=int(round((n8+n12)/2))
                new_count=max(n0,proposal) # never suppress existing detections
        rec={'group_id':r.group_id,'baseline_count':n0,'grid4':n4,'grid8':n8,
             'grid12':n12,'ratio_8_12':round(max(n8,n12)/max(1,min(n8,n12)),4),
             'full_gate':full_label,'full_raw':raw_full,'local_yes':local_yes,
             'panel_complete':panel_complete,'panel_raw':raw_panel,
             'rescue_accepted':approved,'freshneg_count':new_count,'status':'ok',
             'elapsed_sec':round(time.time()-t0,2),'error':''}
    except Exception as e:
        rec={'group_id':r.group_id,'baseline_count':n0,'grid4':n4,'grid8':n8,
             'grid12':n12,'ratio_8_12':None,'full_gate':'ERROR','full_raw':raw_full,
             'local_yes':0,'panel_complete':False,'panel_raw':raw_panel,
             'rescue_accepted':False,'freshneg_count':n0,'status':'error',
             'elapsed_sec':round(time.time()-t0,2),'error':repr(e)}
    rows=upsert_record(rows,rec,GATE_CKPT)
    print(f'[GATE] {i}/{len(triggers)} {r.group_id}: {n0} -> {rec["freshneg_count"]}',
          f'(full={rec["full_gate"]}, local YES={local_yes})')
    if rec['status']=='error':raise RuntimeError('QWEN failed, checkpoint preserved: '+rec['error'])
if len(triggers):assert len(pd.read_csv(GATE_CKPT).query('status=="ok"'))==len(triggers)
print('Qwen stage complete; NO GT was loaded')

In [ ]:
# 17 — SEAL PREDICTIONS FIRST, THEN COMPUTE NEGATIVE SAFETY METRICS
base_df=pd.read_csv(BASE_CKPT)
assert base_df.group_id.is_unique and set(base_df.group_id)==set(work.group_id)
assert base_df.status.eq('ok').all()
pred=work[['group_id','image_name','target','source_class','candidate_type','eval_partition','double_checked']].merge(
    base_df[['group_id','baseline_count']],on='group_id',validate='one_to_one')
if GATE_CKPT.exists():
    gates=pd.read_csv(GATE_CKPT)
    assert gates.group_id.is_unique and set(gates.group_id)==set(triggers.group_id)
    assert gates.status.eq('ok').all()
    pred=pred.merge(gates[['group_id','freshneg_count','rescue_accepted','full_gate','local_yes']],on='group_id',how='left',validate='one_to_one')
    pred['freshneg_count']=pred.freshneg_count.fillna(pred.baseline_count).astype(int)
    pred['rescue_accepted']=pred.rescue_accepted.fillna(False).astype(bool)
else:
    pred['freshneg_count']=pred.baseline_count.astype(int)
    pred['rescue_accepted']=False
    pred['full_gate']='NOT_RUN';pred['local_yes']=0
assert pred.group_id.is_unique and len(pred)==len(work)
assert (pred.baseline_count>=0).all() and (pred.freshneg_count>=0).all()
pred['changed']=pred.freshneg_count.ne(pred.baseline_count)

sealed=OUT/'freshneg_predictions_SEALED.csv'
if sealed.exists():
    previous=pd.read_csv(sealed)
    assert previous[['group_id','freshneg_count']].sort_values('group_id').reset_index(drop=True).equals(
        pred[['group_id','freshneg_count']].sort_values('group_id').reset_index(drop=True)), 'Predictions differ from previous sealed output'
else:
    pred.to_csv(sealed,index=False)
sha=hashlib.sha256(sealed.read_bytes()).hexdigest()
(OUT/'freshneg_prediction_sha256.txt').write_text(sha+'  '+sealed.name+'\n')
print('SEALED SHA256:',sha)

# Ground truth is 0 ONLY for the HUMAN-VERIFIED ABSENT subset.
# We never assume other candidates absent. No GT count was read during inference.
score=pred.copy()
score['gt_count']=0
score['baseline_AE']=score.baseline_count
score['v75_AE']=score.freshneg_count
score['delta_AE']=score.v75_AE-score.baseline_AE
score['new_false_positive']=score.baseline_count.eq(0)&score.freshneg_count.gt(0)
score['more_false_counts']=score.freshneg_count.gt(score.baseline_count)
score.to_csv(OUT/'freshneg_verified_absent_scored.csv',index=False)

def summary(d, tag):
    if len(d)==0:return None
    return {'subset':tag,'n_verified_absent':len(d),
            'n_double_checked':int(d.double_checked.sum()),
            'baseline_MAE':float(d.baseline_count.mean()),'v75_MAE':float(d.freshneg_count.mean()),
            'baseline_FPR':float(d.baseline_count.gt(0).mean()),'v75_FPR':float(d.freshneg_count.gt(0).mean()),
            'baseline_zero':int(d.baseline_count.eq(0).sum()),
            'v75_zero':int(d.freshneg_count.eq(0).sum()),
            'n_new_FP_from_zero':int(d.new_false_positive.sum()),
            'n_increased_counts':int(d.more_false_counts.sum()),
            'n_rescued':int(d.rescue_accepted.sum()),
            'extra_false_counts_sum':int(d.delta_AE.sum())}
rows=[]
for key,sub in [('ALL',score)]+[(x,score[score.candidate_type.eq(x)]) for x in sorted(score.candidate_type.unique())]+[(x,score[score.eval_partition.eq(x)]) for x in sorted(score.eval_partition.unique())]:
    x=summary(sub,key)
    if x:rows.append(x)
metrics=pd.DataFrame(rows)
metrics.to_csv(OUT/'freshneg_safety_metrics.csv',index=False)
score.loc[score.changed].sort_values('delta_AE',ascending=False).to_csv(OUT/'freshneg_changed_cases.csv',index=False)
print(metrics.to_string(index=False))
print('ANNOTATION: single-human-reviewer labels, no independent consensus; label noise possible.')
print('CRITICAL: This measures safety of FROZEN V7.5 rescue relative to CountGD++ baseline on NEW reviewed negatives;')
print('NOT the full V7 + V7.5 pipeline FPR. Do not report as V7.5 end-to-end unless V7 router has also been executed.')

In [ ]:
# 18 — EXPORT RESULT AND CHECKPOINT; NEVER overwrite old validation reports
from IPython.display import FileLink, display
finalzip=ROOT/'V75_FreshNeg_Safety_Results.zip'
with zipfile.ZipFile(finalzip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p.suffix.lower() in ('.csv','.json','.txt','.jpg'):
            z.write(p,p.name)
    p=ROOT/'V75_FreshNeg_Partial_Checkpoint.zip'
    if p.exists():z.write(p,p.name)
print('DONE:',finalzip,'size MB:',round(finalzip.stat().st_size/1e6,2))
display(FileLink(str(finalzip)))